# ViralSense EDA
Exploration only: no modelling decisions are fit here. Run after `python -m viralsense.data.prepare`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from viralsense.utils import load_config
import viralsense.plots  # shared style
cfg = load_config()
posts = pd.read_parquet(f"{cfg['paths']['processed']}/posts.parquet")
posts.shape, posts['account'].nunique()

## Sample composition

In [ ]:
posts.groupby('split').agg(posts=('post_id','size'), accounts=('account','nunique'))

In [ ]:
pd.crosstab(posts['follower_tier'], posts['label'], normalize='index').round(3)

## Engagement rate distribution by follower tier

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))
for t, g in posts.groupby('follower_tier'):
    ax.hist(np.log10(g['engagement_rate'] + 1e-6), bins=60, histtype='step', linewidth=2, label=f'tier {t}')
ax.set_xlabel('log10 engagement rate'); ax.legend(); plt.show()

## Posting hour (UTC) and weekday

In [ ]:
posts['hour'] = posts['timestamp'].dt.hour
posts.groupby('hour')['label'].apply(lambda s: (s != 'Low').mean()).plot(kind='bar', figsize=(8,3), title='share Moderate or Viral by UTC hour'); plt.show()

In [ ]:
posts['timestamp'].dt.day_name().value_counts()

## Captions

In [ ]:
cap = posts['caption'].fillna('')
pd.DataFrame({'length': cap.str.len(), 'hashtags': cap.str.count(r'#\w+')}).describe().round(1)

## History feature coverage
Share of posts with no earlier post from the same account (feature is NaN).

In [ ]:
posts['acct_past_median_er'].isna().mean()